# Risk-Aware Patient–Trial Matching
**Integrating Trial Outcome Risk Scoring into Semantic Retrieval for Clinical Trial Ranking**

Deekshitha R — Twinkle Sahu

Master's Thesis Project — Biomedical AI / Clinical NLP

## 1. Environment Setup

In [ ]:
# All dependencies
# pip install sentence-transformers faiss-cpu xgboost scikit-learn
# pip install imbalanced-learn shap fastapi uvicorn rank_bm25 pandas numpy

## 2. Feature Engineering (Stage 3 — Risk Classifier)

In [ ]:
import os
import ast
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

RAW_DIR = '/dgxa_home/se25mbds002/trial_match/data/raw'
OUT_DIR = '/dgxa_home/se25mbds002/trial_match/data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# ── 1. Load all splits ────────────────────────────────────────────────────────
dfs = {}
for phase in ['I', 'II', 'III']:
    for split in ['train', 'valid', 'test']:
        key = f'{phase}_{split}'
        dfs[key] = pd.read_csv(f'{RAW_DIR}/phase_{phase}_{split}.csv')
        dfs[key]['phase_label'] = phase
        dfs[key]['split'] = split

all_data = pd.concat(dfs.values(), ignore_index=True)
print(f'Total rows: {len(all_data)}')

# ── 2. Safe list parser ───────────────────────────────────────────────────────
def safe_parse(val):
    try:
        return ast.literal_eval(val)
    except:
        return []

# ── 3. Phase encoding ─────────────────────────────────────────────────────────
phase_map = {'I': 1, 'II': 2, 'III': 3}
all_data['phase_ordinal'] = all_data['phase_label'].map(phase_map)

# ── 4. Status encoding ────────────────────────────────────────────────────────
# terminated/withdrawn/suspended = high failure signal
all_data['is_terminated'] = all_data['status'].isin(
    ['terminated', 'withdrawn', 'suspended']).astype(int)

# ── 5. Drug count ─────────────────────────────────────────────────────────────
all_data['drug_count'] = all_data['drugs'].apply(
    lambda x: len(safe_parse(x)))

# ── 6. Disease count ──────────────────────────────────────────────────────────
all_data['disease_count'] = all_data['diseases'].apply(
    lambda x: len(safe_parse(x)))

# ── 7. ICD code count ─────────────────────────────────────────────────────────
def count_icds(val):
    try:
        outer = ast.literal_eval(val)
        total = 0
        for item in outer:
            inner = ast.literal_eval(item) if isinstance(item, str) else item
            total += len(inner) if isinstance(inner, list) else 1
        return total
    except:
        return 0

all_data['icd_count'] = all_data['icdcodes'].apply(count_icds)

# ── 8. Criteria length ────────────────────────────────────────────────────────
all_data['criteria_length'] = all_data['criteria'].fillna('').apply(len)

# ── 9. Has SMILES (drug chemical info available) ──────────────────────────────
all_data['has_smiles'] = all_data['smiless'].apply(
    lambda x: int(len(safe_parse(x)) > 0))

# ── 10. Study year ────────────────────────────────────────────────────────────
all_data['study_year'] = pd.to_datetime(
    all_data['study_first_submitted_date'], errors='coerce').dt.year
all_data['study_year'] = all_data['study_year'].fillna(
    all_data['study_year'].median())

# ── 11. Top disease encoding (target encode by failure rate) ──────────────────
def get_first_disease(val):
    parsed = safe_parse(val)
    return parsed[0].lower().strip() if parsed else 'unknown'

all_data['primary_disease'] = all_data['diseases'].apply(get_first_disease)

# Target encode: mean label per disease (on train only to avoid leakage)
train_mask = all_data['split'] == 'train'
disease_failure_rate = (
    all_data[train_mask]
    .groupby('primary_disease')['label']
    .mean()
    .rename('disease_target_enc')
)
all_data = all_data.merge(
    disease_failure_rate, on='primary_disease', how='left')
global_mean = all_data[train_mask]['label'].mean()
all_data['disease_target_enc'] = all_data['disease_target_enc'].fillna(global_mean)

# ── 12. Final feature matrix ──────────────────────────────────────────────────
FEATURE_COLS = [
    'phase_ordinal',
    'is_terminated',
    'drug_count',
    'disease_count',
    'icd_count',
    'criteria_length',
    'study_year',
    'disease_target_enc',
]

print('\n=== Feature matrix sample ===')
print(all_data[FEATURE_COLS].head())
print('\n=== Null check ===')
print(all_data[FEATURE_COLS].isnull().sum())
print('\n=== Label distribution ===')
print(all_data['label'].value_counts(normalize=True).round(3))

# ── 13. Save train/val/test splits ────────────────────────────────────────────
for split in ['train', 'valid', 'test']:
    mask = all_data['split'] == split
    X = all_data[mask][FEATURE_COLS].values
    y = all_data[mask]['label'].values
    nctids = all_data[mask]['nctid'].values
    np.save(f'{OUT_DIR}/X_{split}.npy', X)
    np.save(f'{OUT_DIR}/y_{split}.npy', y)
    np.save(f'{OUT_DIR}/nctids_{split}.npy', nctids)
    print(f'Saved {split}: X={X.shape}, y={y.shape}')

# Save feature names and disease encoder for API use
with open(f'{OUT_DIR}/feature_cols.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)
with open(f'{OUT_DIR}/disease_target_enc.pkl', 'wb') as f:
    pickle.dump(disease_failure_rate, f)

# Save full dataframe with criteria text for retrieval stage
all_data[['nctid', 'criteria', 'title', 'split']].drop_duplicates(
    subset='nctid', keep='first').to_csv(
    f'{OUT_DIR}/trials_for_retrieval.csv', index=False)

print('\nFeature engineering complete.')


## 3. XGBoost Classifier Training

In [ ]:
import numpy as np
import pickle
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.calibration import CalibratedClassifierCV
import shap

OUT_DIR = '/dgxa_home/se25mbds002/trial_match/data/processed'
MODEL_DIR = '/dgxa_home/se25mbds002/trial_match/models'
import os
os.makedirs(MODEL_DIR, exist_ok=True)

# Load data
X_train = np.load(f'{OUT_DIR}/X_train.npy')
y_train = np.load(f'{OUT_DIR}/y_train.npy')
X_valid = np.load(f'{OUT_DIR}/X_valid.npy')
y_valid = np.load(f'{OUT_DIR}/y_valid.npy')
X_test  = np.load(f'{OUT_DIR}/X_test.npy')
y_test  = np.load(f'{OUT_DIR}/y_test.npy')

with open(f'{OUT_DIR}/feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

print(f'Train: {X_train.shape}, Valid: {X_valid.shape}, Test: {X_test.shape}')
print(f'Features: {feature_cols}')

# Combine train+valid for final training (use valid only for CV)
X_trainval = np.vstack([X_train, X_valid])
y_trainval = np.concatenate([y_train, y_valid])

# Pipeline: SMOTE inside CV folds
pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('clf', XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=4
    ))
])

# 5-fold cross-validation on train+valid
print('\nRunning 5-fold CV...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipeline, X_trainval, y_trainval,
    cv=cv, scoring='roc_auc', n_jobs=1
)
print(f'CV ROC-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'Per-fold:   {[round(s,4) for s in cv_scores]}')

# Train final model on full train+valid
print('\nTraining final model...')
pipeline.fit(X_trainval, y_trainval)

# Evaluate on held-out test set
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)
test_auc = roc_auc_score(y_test, y_prob)
print(f'\nTest ROC-AUC: {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

# Save model
with open(f'{MODEL_DIR}/xgb_pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)
print(f'Model saved to {MODEL_DIR}/xgb_pipeline.pkl')

# SHAP values on test set
print('\nComputing SHAP values...')
explainer = shap.TreeExplainer(pipeline.named_steps['clf'])
X_test_transformed = pipeline.named_steps['smote']
# SHAP on raw test features (no SMOTE at inference)
shap_values = explainer.shap_values(X_test)
print('SHAP mean absolute values per feature:')
for fname, val in sorted(
    zip(feature_cols, np.abs(shap_values).mean(axis=0)),
    key=lambda x: -x[1]
):
    print(f'  {fname:30s}: {val:.4f}')

# Save explainer
with open(f'{MODEL_DIR}/shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
print('\nSHAP explainer saved.')
print('\nDone.')


## 4. BM25 Baseline (Stage 1 — Retrieval Baseline)

In [ ]:
import os
import ast
import pandas as pd
import numpy as np
import pickle
import json
from rank_bm25 import BM25Okapi
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

PROCESSED_DIR = '/dgxa_home/se25mbds002/trial_match/data/processed'
MODEL_DIR = '/dgxa_home/se25mbds002/trial_match/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Load trials with criteria text
trials_df = pd.read_csv(f'{PROCESSED_DIR}/trials_for_retrieval.csv')
trials_df = trials_df.dropna(subset=['criteria']).reset_index(drop=True)
print(f'Total trials for retrieval index: {len(trials_df)}')

# Tokenize
stop_words = set(stopwords.words('english'))

def tokenize(text):
    tokens = word_tokenize(str(text).lower())
    return [t for t in tokens if t.isalnum() and t not in stop_words]

print('Tokenizing corpus...')
tokenized_corpus = [tokenize(text) for text in trials_df['criteria']]
print(f'Tokenization done. Sample: {tokenized_corpus[0][:10]}')

# Build BM25 index
print('Building BM25 index...')
bm25 = BM25Okapi(tokenized_corpus)

# Save index and mapping
with open(f'{MODEL_DIR}/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

trials_df[['nctid', 'title']].to_csv(
    f'{PROCESSED_DIR}/retrieval_index_map.csv', index=True)

print(f'BM25 index saved.')
print(f'Index size: {len(tokenized_corpus)} trials')

# Quick sanity check - query with a sample
sample_query = "lung cancer chemotherapy eligibility adult patients"
tokens = tokenize(sample_query)
scores = bm25.get_scores(tokens)
top5_idx = scores.argsort()[-5:][::-1]
print(f'\nSanity check - top 5 results for: "{sample_query}"')
for i, idx in enumerate(top5_idx):
    print(f'  {i+1}. [{trials_df.iloc[idx]["nctid"]}] '
          f'{str(trials_df.iloc[idx]["title"])[:70]} '
          f'(score: {scores[idx]:.3f})')

print('\nBM25 baseline ready.')


## 5. Retrieval Evaluation Framework

In [ ]:
import numpy as np
import pandas as pd
import pickle
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

PROCESSED_DIR = '/dgxa_home/se25mbds002/trial_match/data/processed'
MODEL_DIR = '/dgxa_home/se25mbds002/trial_match/models'

stop_words = set(stopwords.words('english'))

def tokenize(text):
    tokens = word_tokenize(str(text).lower())
    return [t for t in tokens if t.isalnum() and t not in stop_words]

def dcg_at_k(relevances, k):
    relevances = np.array(relevances[:k], dtype=float)
    if len(relevances) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(relevances) + 2))
    return np.sum(relevances / discounts)

def ndcg_at_k(retrieved_ids, relevant_ids, k=10):
    relevant_set = set(relevant_ids)
    relevances = [1 if nid in relevant_set else 0 for nid in retrieved_ids[:k]]
    ideal = sorted(relevances, reverse=True)
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(ideal, k)
    return dcg / idcg if idcg > 0 else 0.0

def mrr_at_k(retrieved_ids, relevant_ids, k=10):
    relevant_set = set(relevant_ids)
    for i, nid in enumerate(retrieved_ids[:k]):
        if nid in relevant_set:
            return 1.0 / (i + 1)
    return 0.0

def evaluate_bm25(k=10):
    # Load BM25 index
    with open(f'{MODEL_DIR}/bm25_index.pkl', 'rb') as f:
        bm25 = pickle.load(f)

    index_map = pd.read_csv(f'{PROCESSED_DIR}/retrieval_index_map.csv')

    # Load test trials as queries
    # For each test trial, use its criteria as query
    # Relevant = same nctid (self-retrieval sanity check)
    # Real evaluation needs patient-trial relevance labels
    test_nctids = np.load(f'{PROCESSED_DIR}/nctids_test.npy', allow_pickle=True)

    # Build nctid -> index mapping
    nctid_to_idx = {row['nctid']: idx for idx, row in index_map.iterrows()}

    # Load full trials for criteria text
    trials_df = pd.read_csv(f'{PROCESSED_DIR}/trials_for_retrieval.csv')
    nctid_to_criteria = dict(zip(trials_df['nctid'], trials_df['criteria']))

    ndcg_scores, mrr_scores = [], []
    missing = 0

    for nctid in test_nctids[:200]:  # sample 200 for speed
        if nctid not in nctid_to_criteria or nctid not in nctid_to_idx:
            missing += 1
            continue

        query_text = nctid_to_criteria[nctid]
        tokens = tokenize(str(query_text))
        scores = bm25.get_scores(tokens)
        top_k_idx = scores.argsort()[-k:][::-1]
        retrieved_ids = index_map.iloc[top_k_idx]['nctid'].tolist()

        # Ground truth: the trial itself (self-retrieval)
        relevant_ids = [nctid]

        ndcg_scores.append(ndcg_at_k(retrieved_ids, relevant_ids, k))
        mrr_scores.append(mrr_at_k(retrieved_ids, relevant_ids, k))

    print(f'Evaluated {len(ndcg_scores)} queries ({missing} skipped)')
    print(f'BM25 NDCG@{k}: {np.mean(ndcg_scores):.4f}')
    print(f'BM25  MRR@{k}: {np.mean(mrr_scores):.4f}')
    return np.mean(ndcg_scores), np.mean(mrr_scores)

if __name__ == '__main__':
    print('=== BM25 Baseline Evaluation ===')
    ndcg, mrr = evaluate_bm25(k=10)
    print(f'\nFinal: NDCG@10={ndcg:.4f}, MRR@10={mrr:.4f}')
    print('\nNote: self-retrieval evaluation — real evaluation')
    print('requires patient-trial relevance labels.')


## 6. PubMedBERT Bi-Encoder + FAISS Index (Stage 1)
*This stage was run on Google Colab (Tesla T4 GPU)*

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss, os

BASE_LOCAL = '/content/trial_match'
model = SentenceTransformer('microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract')
criteria_texts = df['criteria'].fillna('').tolist()
print(f'Encoding {len(criteria_texts)} trials...')

all_embeddings = []
chunk_size = 2000
for start in range(0, len(criteria_texts), chunk_size):
    chunk = criteria_texts[start:start+chunk_size]
    embs = model.encode(chunk, batch_size=128,
                        show_progress_bar=True,
                        normalize_embeddings=True)
    all_embeddings.append(embs)
    np.save(f'{BASE_LOCAL}/data/processed/embeddings_chunk_{start}.npy', embs)
    print(f'Saved chunk {start}:{start+len(chunk)}, shape={embs.shape}')

embeddings = np.vstack(all_embeddings)
print(f'Final embeddings shape: {embeddings.shape}')
np.save(f'{BASE_LOCAL}/data/processed/trial_embeddings.npy', embeddings)

# Build FAISS index
d = embeddings.shape[1]
quantizer = faiss.IndexFlatIP(d)
index = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_INNER_PRODUCT)
index.train(embeddings)
index.add(embeddings)
index.nprobe = 10
print(f'Index total vectors: {index.ntotal}')
faiss.write_index(index, f'{BASE_LOCAL}/data/processed/trial_index.faiss')

## 7. Cross-Encoder Reranking (Stage 2)

In [ ]:
from sentence_transformers import CrossEncoder
import numpy as np

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

query = "lung cancer chemotherapy adult patients stage IV"
query_emb = model.encode([query], normalize_embeddings=True)
scores, indices = index.search(query_emb, 20)

candidates = []
for idx in indices[0]:
    if idx < len(df):
        row = df.iloc[idx]
        candidates.append({
            'nctid': row['nctid'],
            'title': str(row['title']),
            'criteria': str(row['criteria'])[:512],
            'idx': int(idx)
        })

pairs = [(query, c['criteria']) for c in candidates]
ce_scores = reranker.predict(pairs)
ranked = sorted(enumerate(candidates), key=lambda x: ce_scores[x[0]], reverse=True)

print('Top 10 after cross-encoder reranking:')
for rank, (i, c) in enumerate(ranked[:10]):
    print(f'  {rank+1}. [{c["nctid"]}] {c["title"][:65]} (CE score: {ce_scores[i]:.3f})')

## 8. Final Ranking Formula

In [ ]:
import numpy as np

def compute_final_score(relevance_scores, sae_risks):
    min_r, max_r = relevance_scores.min(), relevance_scores.max()
    norm_relevance = (relevance_scores - min_r) / (max_r - min_r + 1e-9)
    return norm_relevance * (1 - sae_risks)

# Example
rel_scores = np.array([0.9, 0.8, 0.7])
sae_risks  = np.array([0.2, 0.5, 0.1])
final = compute_final_score(rel_scores, sae_risks)
print('Relevance:', rel_scores)
print('SAE Risk: ', sae_risks)
print('Final:    ', final)

## 9. FastAPI Service

In [ ]:
# Run with: uvicorn src.api.main:app --host 0.0.0.0 --port 8000

import os
import pickle
import numpy as np
import pandas as pd
import faiss
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer, CrossEncoder
import shap

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = '/dgxa_home/se25mbds002/trial_match'
PROCESSED = f'{BASE}/data/processed'
MODELS = f'{BASE}/models'

# ── Load all artifacts at startup ─────────────────────────────────────────────
print('Loading artifacts...')

# Trials dataframe
trials_df = pd.read_csv(f'{PROCESSED}/trials_for_retrieval.csv')
print(f'Trials loaded: {len(trials_df)}')

# FAISS index
index = faiss.read_index(f'{PROCESSED}/trial_index.faiss')
index.nprobe = 10
print(f'FAISS index loaded: {index.ntotal} vectors')

# Bi-encoder
biencoder = SentenceTransformer(
    'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract')
print('Bi-encoder loaded.')

# Cross-encoder
crossencoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
print('Cross-encoder loaded.')

# XGBoost pipeline
with open(f'{MODELS}/xgb_pipeline.pkl', 'rb') as f:
    xgb_pipeline = pickle.load(f)
print('XGBoost pipeline loaded.')

# Feature columns
with open(f'{PROCESSED}/feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

# Disease target encoding map
with open(f'{PROCESSED}/disease_target_enc.pkl', 'rb') as f:
    disease_enc = pickle.load(f)

# SHAP explainer
with open(f'{MODELS}/shap_explainer.pkl', 'rb') as f:
    explainer = pickle.load(f)

global_mean = 0.573  # from training label distribution

print('All artifacts loaded. API ready.')

# ── FastAPI app ───────────────────────────────────────────────────────────────
app = FastAPI(title='Risk-Aware Trial Matching API')

class PatientInput(BaseModel):
    patient_id: str
    age: int
    sex: str
    diagnosis_codes: list[str]
    medications: list[str]
    free_text_summary: str

# ── Helper: build XGBoost features for a trial row ───────────────────────────
def build_features(trial_row, phase_ordinal):
    primary_disease = trial_row.get('primary_disease', 'unknown')
    disease_enc_val = disease_enc.get(primary_disease, global_mean)
    criteria_text = str(trial_row.get('criteria', ''))

    features = {
        'phase_ordinal': phase_ordinal,
        'is_terminated': 0,  # unknown at match time
        'drug_count': 1,
        'disease_count': len(trial_row.get('diagnosis_codes', [])),
        'icd_count': 0,
        'criteria_length': len(criteria_text),
        'study_year': 2020,
        'disease_target_enc': disease_enc_val,
    }
    return [features[col] for col in feature_cols]

# ── Helper: top SHAP features ─────────────────────────────────────────────────
def get_top_shap(shap_row, n=3):
    abs_vals = np.abs(shap_row)
    top_idx = abs_vals.argsort()[-n:][::-1]
    return [
        {'feature': feature_cols[i], 'value': round(float(shap_row[i]), 4)}
        for i in top_idx
    ]

# ── Main endpoint ─────────────────────────────────────────────────────────────
@app.post('/match')
def match(patient: PatientInput):
    # Stage 1: Dense retrieval — top 100
    query_emb = biencoder.encode(
        [patient.free_text_summary],
        normalize_embeddings=True
    )
    scores, indices = index.search(query_emb, 100)

    candidates = []
    for idx, score in zip(indices[0], scores[0]):
        if idx < len(trials_df):
            row = trials_df.iloc[idx]
            candidates.append({
                'nctid': row['nctid'],
                'title': str(row['title']),
                'criteria': str(row['criteria'])[:512],
                'retrieval_score': float(score),
                'idx': int(idx)
            })

    # Stage 2: Cross-encoder reranking — top 20
    pairs = [(patient.free_text_summary, c['criteria']) for c in candidates]
    ce_scores = crossencoder.predict(pairs)
    ranked_idx = np.argsort(ce_scores)[::-1][:20]
    reranked = [(candidates[i], float(ce_scores[i])) for i in ranked_idx]

    # Normalise relevance scores to [0, 1]
    rel_scores = np.array([s for _, s in reranked])
    min_r, max_r = rel_scores.min(), rel_scores.max()
    norm_rel = (rel_scores - min_r) / (max_r - min_r + 1e-9)

    # Stage 3: SAE risk scoring + final ranking
    results = []
    for i, (candidate, rel_score) in enumerate(reranked):
        trial_row = trials_df.iloc[candidate['idx']]
        features = build_features(trial_row, phase_ordinal=2)
        X = np.array([features])

        sae_risk = float(xgb_pipeline.predict_proba(X)[0][1])
        final_score = float(norm_rel[i] * (1 - sae_risk))
        shap_vals = explainer.shap_values(X)[0]
        top_shap = get_top_shap(shap_vals)

        results.append({
            'nctid': candidate['nctid'],
            'title': candidate['title'],
            'relevance_score': round(float(rel_score), 4),
            'sae_risk': round(sae_risk, 4),
            'final_score': round(final_score, 4),
            'shap_top_features': top_shap
        })

    # Sort by final score descending
    results.sort(key=lambda x: x['final_score'], reverse=True)
    return results

@app.get('/health')
def health():
    return {'status': 'ok', 'trials_indexed': index.ntotal}


## 10. Results Summary

In [ ]:
print("=== Stage 3: XGBoost Trial Outcome Classifier ===")
print("CV  ROC-AUC : 0.8859 +/- 0.0077")
print("Test ROC-AUC: 0.8312")
print("Test Accuracy: 80%")
print()
print("=== Top SHAP Features ===")
features = [
    ("is_terminated",    1.4196),
    ("study_year",       1.0998),
    ("disease_target_enc",1.0359),
    ("criteria_length",  0.2179),
    ("phase_ordinal",    0.1533),
]
for f, v in features:
    print(f"  {f:30s}: {v:.4f}")
print()
print("=== Pipeline Output (sample) ===")
print("Top result: NCT01998919")
print("  Title:           A Study of Tarceva (Erlotinib) in Combination With Platinum...")
print("  Relevance Score: 4.9093")
print("  Risk Score:      0.8113")
print("  Final Score:     0.1650")